# Coder 에이전트 구축

Navigator 에이전트가 브라우저를 탐색하여 지도를 그리는 역할이었다면, 이번에 만들 **Coder 에이전트**는 파이썬 코드를 직접 작성하고 실행하며 디버깅하는 '개발자' 역할을 수행합니다.

단순히 코드를 짜서 글자로만 알려주는 LLM이 아니라,
**코드를 [.py](cci:7://file:///d:/agent_research/github-repos/AAWS_project/app/client.py:0:0-0:0) 파일로 저장하고 → 직접 실행해보고 → 에러가 나면 스스로 고치는(디버깅)** 능력을 갖춘 완벽한 자율 에이전트입니다.

### 핵심 아이디어 1: Python Execution Tool
에이전트에게 "코드를 실행하는 능력"을 주기 위해 `subprocess` 모듈을 활용하여 지정된 격리 공간(`code_artifacts/`) 안에서만 파이썬 스크립트를 작성하고 실행하는 전용 도구를 만들어 제공합니다.

### 핵심 아이디어 2: File Search Middleware (에이전트의 시야 확장)
개발자가 코드를 짜려면 현재 작업 폴더에 어떤 파일들이 있는지 파악해야 합니다.
LangChain의 `FilesystemFileSearchMiddleware`를 추가해, 에이전트가 `code_artifacts/` 폴더 내의 파일들을 검색하고 읽어볼 수 있는 도구를 자동으로 쥐여줍니다.


In [ ]:
import os, sys
from dotenv import load_dotenv

load_dotenv(override=True)
#load_dotenv("/mnt/d/agent_research/github-repos/home/AAWS_project/.env", override=True)

# PROJECT_ROOT를 .env에서 읽기 (없으면 현재 디렉토리)
project_root = os.getenv("PROJECT_ROOT", os.getcwd())

# 프로젝트 루트가 유효하지 않으면, 현재 위치에서 상위로 찾기
if not os.path.exists(os.path.join(project_root, "app")):
    # 상위 디렉토리 탐색
    current = os.getcwd()
    for _ in range(5):
        if os.path.exists(os.path.join(current, "app")):
            project_root = current
            break
        current = os.path.dirname(current)

# Working Directory 설정
os.chdir(project_root)
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import app

print(f"✅ Project Root: {project_root}")
print(f"✅ Working Directory: {os.getcwd()}")

In [ ]:
import os
import subprocess
from langchain.tools import tool

# ==========================================
# 1. 🛠️ 파이썬 코드 실행 공간 및 도구 (Tool)
# ==========================================

# 작업 및 실행 파일들이 모일 대상 디렉토리 (없으면 자동 생성)
ARTIFACT_DIR = os.path.join(os.getenv("PROJECT_ROOT", os.getcwd()), "code_artifacts")
os.makedirs(ARTIFACT_DIR, exist_ok=True)

@tool(parse_docstring=True)
def create_new_file(filepath: str, content: str) -> str:
    """새로운 파이썬 파일이나 텍스트 문서를 생성하고 초기 내용을 통째로 작성합니다.
    이미 같은 이름의 파일이 존재할 경우 완전히 덮어씁니다! 기존 파일의 일부만 수정하려면 반드시 edit_code_file을 사용하세요.
    
    Args:
        filepath: 생성할 파일명 (예: main.py)
        content: 파일에 들어갈 초기 파이썬 코드 전체 스크립트
    """
    safe_filepath = os.path.join(ARTIFACT_DIR, os.path.basename(filepath))
    
    with open(safe_filepath, "w", encoding="utf-8") as f:
        f.write(content)
        
    return f"[Success] '{filepath}' 파일이 성공적으로 생성되었습니다."
@tool(parse_docstring=True)
def read_code_file(filepath: str, start_line: int = 1, end_line: int = None) -> str:
    """지정된 파일의 내용을 줄 번호(Line number)와 함께 읽어옵니다.
    코드를 수정하기 전, 정확히 몇 번째 줄을 수정해야 할지 파악하기 위해 반드시 먼저 사용하세요.
    
    Args:
        filepath: 읽을 파일의 경로 (파일명만 입력하면 code_artifacts 폴더 안에서 찾습니다)
        start_line: 읽기 시작할 줄 번호 (기본값: 1)
        end_line: 읽기를 끝낼 줄 번호 (입력하지 않으면 끝까지 읽음)
    """
    safe_filepath = os.path.join(ARTIFACT_DIR, os.path.basename(filepath))
    if not os.path.exists(safe_filepath):
        return f"[Error] 파일이 존재하지 않습니다: {safe_filepath}"
        
    with open(safe_filepath, "r", encoding="utf-8") as f:
        lines = f.readlines()
        
    end = end_line if end_line else len(lines)
    start = max(1, start_line)
    
    if start > len(lines):
        return "[Error] start_line이 파일의 전체 줄 수보다 큽니다."
        
    output = []
    for i in range(start - 1, min(end, len(lines))):
        output.append(f"{i + 1:03d} | {lines[i].rstrip()}")
        
    return "\n".join(output)
@tool(parse_docstring=True)
def edit_code_file(filepath: str, start_line: int, end_line: int, new_content: str) -> str:
    """기존 파이썬 파일의 특정 줄(Line) 구간만 새로운 내용으로 교체합니다.
    파일 전체를 덮어쓰는 대신, 수정이 필요한 좁은 범위의 코드만 아주 효율적으로 외과수술처럼 변경하세요.
    
    Args:
        filepath: 수정할 기존 파일명
        start_line: 교체를 시작할 기존 줄 번호 (이 줄부터 덮어써짐)
        end_line: 교체를 끝낼 기존 줄 번호 (이 줄까지 덮어써짐)
        new_content: 해당 구간에 통째로 새로 들어갈 코드 내용
    """
    safe_filepath = os.path.join(ARTIFACT_DIR, os.path.basename(filepath))
    if not os.path.exists(safe_filepath):
        return f"[Error] 파일이 존재하지 않습니다. create_new_file을 먼저 사용하세요: {safe_filepath}"
        
    with open(safe_filepath, "r", encoding="utf-8") as f:
        lines = f.readlines()
        
    if start_line < 1 or end_line > len(lines) or start_line > end_line:
        return f"[Error] 잘못된 줄 번호 범위입니다. (현재 파일 총 라인 수: {len(lines)})"
        
    new_lines = [line + "\n" for line in new_content.split("\n")]
    updated_lines = lines[:start_line-1] + new_lines + lines[end_line:]
    
    with open(safe_filepath, "w", encoding="utf-8") as f:
        f.writelines(updated_lines)
        
    return f"[Success] {filepath} 파일의 {start_line}~{end_line} 라인이 성공적으로 교체되었습니다."
@tool(parse_docstring=True)
def run_python_script(filepath: str, script_args: str = "") -> str:
    """저장된 파이썬 스크립트를 즉시 독립된 프로세스에서 실행하고 그 결과(출력 및 에러 로그)를 반환합니다.
    코드를 생성하거나 수정한 직후에는 반드시 이 툴을 호출하여 에러 없이 의도대로 돌아가는지 검증하세요.
    
    Args:
        filepath: 실행할 파이썬 파일명 (예: main.py)
        script_args: 실행 시 덧붙일 커맨드라인 인자 (선택사항)
    """
    safe_filename = os.path.basename(filepath)
    full_path = os.path.join(ARTIFACT_DIR, safe_filename)
    
    if not os.path.exists(full_path):
         return f"[Error] 실행할 파일이 존재하지 않습니다: {safe_filename}"
         
    command = ["python", safe_filename]
    if script_args:
        command.extend(script_args.split())
        
    print(f"\n🚀 [Coder Run] '{safe_filename}' 실행 중...")
    
    try:
        result = subprocess.run(
            command, 
            cwd=ARTIFACT_DIR,
            capture_output=True, 
            text=True, 
            timeout=120 
        )
        
        output = result.stdout
        if result.stderr:
            output += f"\n[Error Output]\n{result.stderr}\n[Action Required] 에러 로그의 줄 번호를 확인하고, read_code_file과 edit_code_file로 위 에러를 해결하세요."
            
        if not output.strip():
            output = "[System] 코드가 에러 없이 정상 실행되었으나, 터미널에 출력(print)된 내용이 없습니다."
            
        return output
        
    except subprocess.TimeoutExpired:
        return "[Error] 실행 시간(120초)을 초과했습니다. 무한 루프(while True 등)나 블로킹 처리를 확인하고 수정하세요."
    except Exception as e:
        return f"[System Error] 코드 실행 오류 발생: {str(e)}"


### Coder 에이전트(Agent) 조립하기
LangChain의 [create_agent]를 활용해 파이썬 실행 도구와 파일 검색 미들웨어를 장착한 에이전트를 선언합니다.


In [ ]:
from dataclasses import dataclass
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import FilesystemFileSearchMiddleware
# ==========================================
# 2. 🤖 Coder 에이전트 정의 (create_agent)
# ==========================================
@dataclass
class CoderContext:
    pass
coder_model = init_chat_model("google_genai:gemini-flash-latest", temperature=0.2)
checkpointer = InMemorySaver()
CODER_SYSTEM_PROMPT = """
당신은 최고 수준의 시니어 파이썬 소프트웨어 엔지니어(Senior SWE)입니다.
당신의 임무는 요구사항에 맞춰 코드를 견고하게 설계, 작성, 테스트, 그리고 스스로 디버깅하여 오류 없이 작동하도록 완성하는 것입니다.
[시니어 엔지니어링 행동 지침]
1. 분리된 기능의 영리한 사용:
   - 당신에게는 "코드 저장(create/edit)" 권한과 "코드 실행(run)" 권한이 완전히 분리된 4개의 도구가 주어집니다. 이를 적재적소에 사용하세요.
2. 정밀한 외과 수술적 수정 (Surgical Edit):
   - 파일을 처음 만들 때는 `create_new_file`을 쓰세요.
   - 단, 기존 파일에 에러가 났거나 기능을 덧붙일 때는 절대로 전체 코드를 다시 작성하지 마세요.
   - 반드시 `read_code_file`을 호출해 코드를 라인 번호와 함께 읽어들인 뒤, 에러가 발생한 지점을 찾아내고 `edit_code_file`을 이용해 특정 라인만 교체하세요.
3. 검증 없는 코딩은 없다 (Test-Driven):
   - 코드를 생성했거나 수정했다면, 반드시 그 직후에 `run_python_script`로 실행해봐야 합니다.
   - [Error Output]이 나오면 에러 메시지와 줄 번호를 분석하여 위 2번 지침을 즉시 반복하세요.
4. 한계 인정 및 에스컬레이션:
   - 동일한 에러가 3회 이상 반복되면 시도를 중단하고 유저에게 보고하세요.
5. 코드 품질:
   - PEP 8 스타일을 준수하고, 모든 함수에 한 줄 docstring을 작성하세요.
"""
coder_agent = create_agent(
    model=coder_model,
    system_prompt=CODER_SYSTEM_PROMPT,
    context_schema=CoderContext,
    tools=[create_new_file, read_code_file, edit_code_file, run_python_script],
    middleware=[
        FilesystemFileSearchMiddleware(
            root_path=ARTIFACT_DIR,
            use_ripgrep=True,
            max_file_size_mb=10,
        )
    ],
    checkpointer=checkpointer
)

In [ ]:
ARTIFACT_DIR

### 🧪 테스트: 도구 분리 + 자가 디버깅 능력 검증
이전 버전의 Coder는 코드를 통째로 작성하고 실행하는 단일 도구만 사용했습니다.  
하지만 에러가 나면 **전체 코드를 처음부터 다시 작성**해야 하는 비효율이 있었습니다.
새로운 Coder는 **4개의 분리된 도구**를 사용합니다:
1. `create_new_file` — 파일 최초 생성
2. `read_code_file` — 줄 번호와 함께 코드 읽기
3. `edit_code_file` — 특정 줄만 외과수술처럼 교체
4. `run_python_script` — 저장된 파일 실행
아래 미션을 통해 Coder가 **create → run → 에러 발견 → read → edit → run** 루프를 
자율적으로 수행하는지 확인합니다.

In [ ]:
from langchain_core.messages import HumanMessage

config = {"configurable": {"thread_id": "coder_session_surgical"}}
context = CoderContext()

# 3단계 미션: 생성 → 디버깅 → 기능 확장
query = (
    "1. 'calculator.py'를 파이썬으로 만들어줘. "
    "add, subtract 함수를 가진 Calculator 클래스를 짜고 "
    "출력으로 'Calculator Created'를 찍게 한 뒤 실행해봐.\n"
    "2. 실행을 확인한 뒤에는 calculator.py의 특정 라인을 수정(edit_code_file 사용)해서 "
    "multiply와 divide 함수를 추가해 봐. 전체 파일을 새로 덮어쓰면 절대 안 돼!\n"
    "3. 마지막으로 수정된 파일을 다시 한번 실행해보고 오류가 없으면 결과를 보고해줘."
)
print(f"👤 User Mission:\n{query}\n")
print("-" * 50)

response = await coder_agent.ainvoke(
    {"messages": [HumanMessage(query)]},
    config=config, context=context
)

print(f"\n🤖 Assistant: {response['messages'][-1].content}")


In [ ]:
for m in response["messages"]:
    m.pretty_print()

In [ ]:
    query2 = "좋아, 그럼 방금 생성한 코드 파일 안에 무슨 내용이 적혀있는지 정확히 확인하고 알려줄래?"
    print(f"👤 User: {query2}\n")
    print("-" * 50)
    
    response2 = await coder_agent.ainvoke(
        {"messages": [HumanMessage(query2)]},
        config=config, context=context
    )
    print(f"\n🤖 Assistant: {response2['messages'][-1].content}")

In [ ]:
for m in response2["messages"]:
    m.pretty_print()

Coder 에이전트에게 1부터 100까지 소수(Prime number)를 구하는 코드를 짜달라고 요청해 보겠습니다. 
과연 직접 계산해서 답을 내는지, 아니면 코드를 짜서 실행시켜 보는지 확인하세요!


In [ ]:
config = {"configurable": {"thread_id": "coder_session_1"}}
context = CoderContext()

query = "1부터 100까지의 숫자 중에서 소수(Prime Number)만 찾아내는 파이썬 코드를 작성하고 실행해서, 나에게 출력 결과만 깔끔하게 보여줘."
print(f"👤 User: {query}\n")
print("-" * 50)

response3 = await coder_agent.ainvoke(
    {"messages": [HumanMessage(query)]},
    config=config,
    context=context
)

print("-" * 50)
print(f"\n🤖 Assistant: {response['messages'][-1].content}")

In [ ]:
for m in response3["messages"]:
    m.pretty_print()

### 실습: Coder 에이전트 앱 서버 연동하기

이제 똑똑한 코딩 비서도 완성되었으니, 기존의 앱 서버(`app`)에 **Coder 에이전트**를 탑재하여 채팅 화면에서 편하게 디버깅을 시켜봅시다! Navigator 에이전트를 추가했을 때와 비슷한 단계를 거칩니다.
